# ReadMe

Kubernetes-Online-End Point - Goal is to:

1. Create a Container for the model
2. Deploy the Model (Obtain API)

Is this way, the deployment is more efficient.

All proprietary source configurations, project identifiers, private endpoint domains, and active security credentials have been scrubbed or replaced with generic environment variable structures (os.getenv).

In [ ]:
import time
import pandas as pd
from azure.ai.ml.entities import (
    AzureMLOnlineInferencingServer,  # type: ignore
    KubernetesOnlineDeployment,
    KubernetesOnlineEndpoint,
    ModelPackage,
)

from shared_utils.mlflow_helpers import (
    get_az_ml_client,
    set_mlflow_tracking_uri,
)

# Initialize the Azure ML client using environment variables
# Ensure AZURE_SUBSCRIPTION_ID, AZURE_RESOURCE_GROUP, and AZURE_WORKSPACE are set in your environment
ml_client = get_az_ml_client()
set_mlflow_tracking_uri(ml_client)

In [ ]:
MODEL_NAME = "structural_monitoring_sensors_v4"
MODEL_LABEL = "latest"

# Fetch model information dynamically from Azure ML registry
azureml_model_info = ml_client.models.get(MODEL_NAME, label=MODEL_LABEL)

print(azureml_model_info)

## Create the Container

In [ ]:
model_package_name = f"pkg-{azureml_model_info.name}-{azureml_model_info.version}"
model_package_name

In [ ]:
# package the model
model_package_version = str(int(time.time()))


package_config = ModelPackage(
    target_environment=model_package_name,
    target_environment_version=model_package_version,
    inferencing_server=AzureMLOnlineInferencingServer(),
)

model_package = ml_client.models.package(
    azureml_model_info.name, azureml_model_info.version, package_config
)

## AKS Deployment of the Model Package
Pre-built containers are also referred to as Environments in Azure ML. For the deployment, we first need to obtain a reference to our newly created environment. Next, we must create a Kubernetes endpoint and specify, among other things, which cluster we want to use. Only after these steps are completed can the actual deployment take place.

In [ ]:
environment = ml_client.environments.get(name=model_package_name, label="latest")
environment

### 2. Create Kubernetes Endpoint with Azure ML Token Authentication

In [ ]:
# Hyphens are required; underscores ("_") are not allowed in the endpoint name.
# Configure endpoint specifications here.
ENDPOINT_NAME = "structural-monitoring-sensors-endpoint"
COMPUTE_CLUSTER_NAME = "k8s-compute-cluster"

endpoint = KubernetesOnlineEndpoint(
    # Dynamically formatting the name is safer, e.g.: azureml_model_info.name.replace("_", "-")
    name=ENDPOINT_NAME,
    compute=COMPUTE_CLUSTER_NAME,
    auth_mode="aml_token",
)

endpoint

In [ ]:
# hier will truthfully be created the endpoint
ml_client.begin_create_or_update(endpoint).result()

### 3. Deployment of the  Model

In [ ]:
# Speecify a characteristic
deployment = KubernetesOnlineDeployment(
    name="version-1-deployment",  # f"version-{azureml_model_info.version}-deployment",
    endpoint_name=endpoint.name,
    environment=environment,
    instance_count=1,
)

deployment

In [ ]:
# execution
ml_client.begin_create_or_update(deployment).result()

In [ ]:
# Multiple deployments can run simultaneously on a single endpoint.
# You can specify the percentage of traffic routed to each deployment.
# By default, a new deployment receives 0% traffic. To activate it,
# the traffic allocation must be explicitly updated to 100%.

# NOTE: Ensure 'deployment' has been defined prior to updating traffic.
# For example: deployment = KubernetesOnlineDeployment(...)

endpoint.traffic = {deployment.name: 100}
ml_client.begin_create_or_update(endpoint).result()

## Test

In [ ]:
# Import test data generated from the training notebook
df_test_csv = pd.read_csv("test_data.csv")

def inputs_to_json(df: pd.DataFrame, params: dict = None) -> dict:
    """Convert DataFrame to a dictionary formatted for ML endpoint requests."""
    data_dict = {
        "input_data": {
            "columns": df.columns.tolist(),
            "index": df.index.astype(str).tolist(),
            "data": df.values.tolist(),
        }
    }

    if params:
        data_dict["params"] = params

    return data_dict

# Prepare the JSON request payload as a dictionary
data_json = inputs_to_json(df_test_csv)

In [ ]:
df_test = pd.DataFrame(
    data_json["input_data"]["data"], columns=data_json["input_data"]["columns"]
)

In [ ]:
import os

API_BASE_URL = os.getenv("API_BASE_URL", "https://api.yourdomain.com")
endpoint_url = f"{API_BASE_URL}/api/user"

# Execute the request using your environment-managed headers
r = requests.get(endpoint_url, headers=headers)

print(f"Status Code: {r.status_code}")
print(f"Response Body: {r.text}")

In [ ]:
import json
import os
import requests

# Convert the previously prepared dictionary payload into a JSON-formatted string byte stream
body = str.encode(json.dumps(data_json))

# CHANGED: Replaced static placeholders with standard environment variables for production readiness
url = os.getenv("AZURE_ENDPOINT_URL", "https://your-endpoint.azureml.ms/score")
api_key = os.getenv("AZURE_ENDPOINT_API_KEY")

if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

# The azureml-model-deployment header will force the request to go to a specific deployment.
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}",
}

try:
    # NOTE: verify=False disables SSL verification. It's okay for local testing 
    # but should generally be True in a production-ready public repository.
    response = requests.post(url=url, data=body, headers=headers, verify=False)
    response.raise_for_status()  # Raises an exception for 4xx or 5xx status codes

    result = response.json()
    print("Prediction Result:", result)

# FIXED: Replaced urllib.error.HTTPError with requests.exceptions.RequestException 
# to match the requests library being used above.
except requests.exceptions.RequestException as error:
    print(f"The request failed: {error}")
    if response is not None:
        print(f"Status Code: {response.status_code}")
        print(f"Response Headers: {response.headers}")
        print(f"Response Body: {response.text}")

# Manage Model

In [ ]:
import os
import requests

# Retrieve credentials and configurations from environment variables
# Ensure these are set in your local system or GitHub Secrets before running
AZURE_SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "your-subscription-id-here")
AZURE_RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "your-resource-group-here")
GRAFANA_WORKSPACE_NAME = os.getenv("AZURE_GRAFANA_NAME", "your-grafana-workspace-here")
AZURE_BEARER_TOKEN = os.getenv("AZURE_BEARER_TOKEN")

if not AZURE_BEARER_TOKEN:
    raise ValueError("Missing 'AZURE_BEARER_TOKEN' environment variable.")

# Construct the API URL using dynamic environment values
url = (
    f"https://management.azure.com/subscriptions/{AZURE_SUBSCRIPTION_ID}/"
    f"resourceGroups/{AZURE_RESOURCE_GROUP}/providers/Microsoft.Dashboard/"
    f"grafana/{GRAFANA_WORKSPACE_NAME}/api/v1/provisioning/alert-rules?api-version=2023-10-01"
)

headers = {
    "Authorization": f"Bearer {AZURE_BEARER_TOKEN}",
    "Content-Type": "application/json"
}

# Execute the request
response = requests.get(url, headers=headers)
print(f"Response Status Code: {response.status_code}")
print(response.text[:500])

In [ ]:
import os
import requests

# Retrieve the API base domain from your local environment 
API_BASE_URL = os.getenv("API_BASE_URL", "https://api.yourdomain.com")
endpoint_url = f"{API_BASE_URL}/api/user"

# Execute the request with your generic or environment-managed headers
r = requests.get(endpoint_url, headers=headers)

print(f"Status Code: {r.status_code}")
print(f"Response Text: {r.text}")

# Test Data

In [ ]:
import numpy as np

x_in = np.array(
    [
        18.950075,
        18.918230,
        18.884970,
        18.850084,
        18.814154,
        18.777748,
        18.740414,
        18.702185,
        18.663054,
        18.623068,
        18.582188,
        18.540958,
        18.499010,
        18.456430,
        18.413172,
        18.369310,
        18.324799,
        18.279465,
        18.233597,
        18.186943,
        18.139494,
        18.091619,
        18.042780,
        17.993666,
        17.943953,
        17.893896,
        17.843494,
        17.793068,
        17.742374,
        17.691765,
        17.641144,
        17.590246,
        17.539612,
        17.489016,
        17.438353,
        17.387711,
        17.336626,
        17.285826,
        17.235130,
        17.184669,
        17.134628,
        17.085127,
        17.035223,
        16.984680,
        16.934879,
        16.885832,
        16.839426,
        16.795004,
        16.751644,
        16.710997,
    ]
)
 
print(limit1)
print(limit2)
print(len(limit2))